In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import swin_t, Swin_T_Weights
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, f1_score
from collections import defaultdict

# ========================
# FIX RANDOMNESS (STABLE RESULTS)
# ========================

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ========================
# SETTINGS (FASTER)
# ========================

DATASET_PATH = "dataset"
SHOTS = [5, 10, 20, 30]
EPOCHS = 18
VAL_PER_CLASS = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# ========================
# TRANSFORM (CONTROLLED)
# ========================

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

dataset = ImageFolder(DATASET_PATH, transform=transform)
num_classes = len(dataset.classes)

print("Classes:", dataset.classes)

# ========================
# FIXED CLASS ORDER
# ========================

class_indices = defaultdict(list)

for idx, (_, label) in enumerate(dataset.samples):
    class_indices[label].append(idx)

for label in class_indices:
    random.shuffle(class_indices[label])

# ========================
# FEW SHOT FUNCTION
# ========================

def run_few_shot(K_SHOT):

    print("\n=================================")
    print(f"        RUNNING {K_SHOT}-SHOT")
    print("=================================")

    train_indices = []
    val_indices = []

    for label in class_indices:
        train_part = class_indices[label][:K_SHOT]
        val_part = class_indices[label][K_SHOT:K_SHOT+VAL_PER_CLASS]

        train_indices.extend(train_part)
        val_indices.extend(val_part)

        print(f"{dataset.classes[label]} → "
              f"{len(train_part)} train | {len(val_part)} val")

    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)

    # ✅ Faster loaders
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    # ========================
    # MODEL
    # ========================

    model = swin_t(weights=Swin_T_Weights.DEFAULT)
    model.head = nn.Linear(model.head.in_features, num_classes)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.features[-3:].parameters():
        param.requires_grad = True

    for param in model.norm.parameters():
        param.requires_grad = True

    for param in model.head.parameters():
        param.requires_grad = True

    model = model.to(device)

    # ✅ Slight regularization
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    # ✅ Controlled LR (boost only for 30-shot)
    lr = 1.5e-4 if K_SHOT < 30 else 2e-4

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS
    )

    best_acc = 0

    # ========================
    # TRAIN LOOP
    # ========================

    for epoch in range(EPOCHS):

        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            # ✅ Boost stability for 30-shot
            if K_SHOT >= 30:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            running_loss += loss.item()

        scheduler.step()

        # ========================
        # VALIDATION
        # ========================

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds) * 100
        f1 = f1_score(all_labels, all_preds, average="macro") * 100

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(),
                       f"swin_{K_SHOT}shot_best.pth")

        print(f"[{K_SHOT}-Shot | Epoch {epoch+1}/{EPOCHS}] "
              f"Loss: {running_loss:.4f} | "
              f"Acc: {acc:.2f}% | "
              f"F1: {f1:.2f}% | "
              f"Best: {best_acc:.2f}%")

    print(f"\nFinal Best Accuracy for {K_SHOT}-Shot: {best_acc:.2f}%")
    return best_acc


# ========================
# RUN ALL SHOTS
# ========================

results = {}

for shot in SHOTS:
    acc = run_few_shot(shot)
    results[shot] = acc

print("\n================ FINAL RESULTS ================")
for k, v in results.items():
    print(f"{k}-Shot → {v:.2f}%")


Using Device: cuda
Classes: ['DominantFollicle', 'Normal', 'PCO']

        RUNNING 5-SHOT
DominantFollicle → 5 train | 100 val
Normal → 5 train | 36 val
PCO → 5 train | 100 val
[5-Shot | Epoch 1/18] Loss: 1.9614 | Acc: 58.90% | F1: 60.16% | Best: 58.90%
[5-Shot | Epoch 2/18] Loss: 1.1440 | Acc: 59.32% | F1: 59.85% | Best: 59.32%
[5-Shot | Epoch 3/18] Loss: 1.0911 | Acc: 69.92% | F1: 73.25% | Best: 69.92%
[5-Shot | Epoch 4/18] Loss: 0.7552 | Acc: 59.32% | F1: 57.88% | Best: 69.92%
[5-Shot | Epoch 5/18] Loss: 0.9226 | Acc: 52.54% | F1: 46.02% | Best: 69.92%
[5-Shot | Epoch 6/18] Loss: 0.5546 | Acc: 64.83% | F1: 63.61% | Best: 69.92%
[5-Shot | Epoch 7/18] Loss: 0.5747 | Acc: 56.36% | F1: 52.45% | Best: 69.92%
[5-Shot | Epoch 8/18] Loss: 0.9191 | Acc: 67.37% | F1: 65.78% | Best: 69.92%
[5-Shot | Epoch 9/18] Loss: 0.5628 | Acc: 63.98% | F1: 62.61% | Best: 69.92%
[5-Shot | Epoch 10/18] Loss: 0.4218 | Acc: 60.59% | F1: 58.42% | Best: 69.92%
[5-Shot | Epoch 11/18] Loss: 0.4084 | Acc: 58.90% | 